# Agents
This notebook is the solution for the lab `Agents.pdf` (Constructing a Goal-Based Agent using an LLM).

**LLM use:** the program in Task 3 was generated with Claude (Anthropic) from the prompt shown there, then run, tested and modified as described.

### Task 1: Understanding the problem

1. **Environment:** a 2D warehouse grid (7 × 21) with shelving (`#`) and free floor (`.`). It is fully observable, deterministic, static, discrete and single-agent.
2. **Goal:** move the vehicle from `S` (row 1, col 1) to `G` (row 1, col 19) without entering any obstacle cell.
3. **Actions:** `Up`, `Down`, `Left`, `Right`, each moving one grid square, and only allowed when the target square is free and inside the map.
4. **Information the agent maintains:** its current position, the map (a model of which cells are blocked), the goal position, and – while planning – the set of positions already explored plus how each was reached (so the route can be reconstructed).
5. **Why goal-based, not simple reflex:** a simple reflex agent maps the current percept straight to an action ("if wall ahead, turn right"). It has no goal and no look-ahead, so it can loop forever or walk into dead ends. Here the right action depends on *where the agent wants to end up*: the agent uses its model of the world to simulate action sequences and picks the one that reaches `G`. Note that `S` and `G` are in the same row, but the direct route along row 1 is blocked at column 6 – a reflex "move toward the goal" rule would get stuck there immediately.

**Think about it – a warehouse twice as large:** BFS still works and still finds a shortest path, but its time and memory grow with the number of reachable cells (doubling each dimension gives about 4× the cells). For much larger warehouses an informed search such as A* with Manhattan distance becomes more appropriate. Other difficulties that appear at scale: multiple vehicles that must avoid each other, shelves or people that move (dynamic environment, so the plan needs re-planning), partial observability from sensors, and non-uniform costs (turning, congestion), which BFS cannot handle – that needs uniform-cost search or A*.

### Task 2: Designing the agent

| Component | Design |
|---|---|
| Environment | The ASCII warehouse map parsed into a 2D grid |
| Current state | Vehicle position `(row, col)` |
| Goal | Position of `G`; goal test `state == goal` |
| Actions | `Up/Down/Left/Right` filtered by a transition model (no walls, stay in bounds) |
| Decision-making component | A search procedure (BFS) that uses the model to find an action sequence reaching the goal, then executes it step by step |

Block diagram (goal-based agent):

```
            +-------------------------------- Agent ---------------------------------+
 percept    |                                                                        |
 (position) |   +-------------+      +---------------------+      +-------------+   |
 ---------->|-->|   State     |----->|  "What happens if I |----->|  Planner    |   |
            |   | (row, col)  |      |   do action a?"     |      |  (BFS over  |   |
            |   +-------------+      |   transition model  |      |   states)   |   |
            |          ^             +---------------------+      +------+------+   |
            |          |                        ^                        |          |
            |          |             +---------------------+             |          |
            |          |             |  Goal: reach G      |-------------+          |
            |          |             +---------------------+             v          |
            |          |                                        +----------------+  |
            |          +----------------------------------------| Action to take |--|---> action
            |                                                   +----------------+  |    (move)
            +------------------------------------------------------------------------+
                                             |
                                             v
                                   Environment (warehouse grid)
```

### Task 3: Prompt engineering

**Prompt used:**

> Write a well-documented Python program implementing a goal-based agent for the warehouse navigation problem shown below.
> (map pasted here)
> S is the start, G is the goal, # is an obstacle and . is free space. The vehicle may move Up, Down, Left or Right, one grid square per move.
> The program should: represent the warehouse as a two-dimensional grid; determine a collision-free path from S to G; avoid all obstacles; print either the path found or a suitable message if no path exists; explain the search algorithm that has been chosen and why it is appropriate.
> Structure it as an agent class with a percept → plan → act loop, so the agent actually executes the plan in the environment step by step.

The final code (after review) is below.

In [1]:
from collections import deque

WAREHOUSE = """
#####################
#S....#............G#
#.##....##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################
"""

MOVES = {"Up": (-1, 0), "Down": (1, 0), "Left": (0, -1), "Right": (0, 1)}


class Warehouse:
    """The environment: a static 2D grid. Also executes actions and checks for collisions."""

    def __init__(self, text):
        self.grid = [list(row) for row in text.strip("\n").splitlines()]
        self.start = self.goal = None
        for r, row in enumerate(self.grid):
            for c, ch in enumerate(row):
                if ch == "S":
                    self.start = (r, c)
                elif ch == "G":
                    self.goal = (r, c)
        self.robot = self.start

    def is_free(self, pos):
        r, c = pos
        return 0 <= r < len(self.grid) and 0 <= c < len(self.grid[r]) and self.grid[r][c] != "#"

    def percept(self):
        return self.robot

    def execute(self, action):
        dr, dc = MOVES[action]
        nxt = (self.robot[0] + dr, self.robot[1] + dc)
        if not self.is_free(nxt):
            raise RuntimeError(f"Collision! {action} from {self.robot} hits an obstacle")
        self.robot = nxt


class GoalBasedAgent:
    """
    Goal-based agent. It keeps a model of the world (the map), a goal, and plans with
    breadth-first search: BFS explores positions in order of distance from the start, so the
    first time it reaches the goal it has found a path with the fewest moves. All moves cost
    the same, so 'fewest moves' = 'shortest path'. It is complete (finds a path if one exists)
    and the map is small, so BFS's memory use is not a problem.
    """

    def __init__(self, world_model, goal):
        self.model = world_model
        self.goal = goal
        self.plan = []
        self.expanded = 0

    def successors(self, state):
        for action, (dr, dc) in MOVES.items():
            nxt = (state[0] + dr, state[1] + dc)
            if self.model.is_free(nxt):
                yield action, nxt

    def search(self, start):
        frontier = deque([start])
        parent = {start: None}                  # state -> (previous state, action)
        self.expanded = 0
        while frontier:
            state = frontier.popleft()
            if state == self.goal:
                actions = []
                while parent[state] is not None:
                    prev, act = parent[state]
                    actions.append(act)
                    state = prev
                return actions[::-1]
            self.expanded += 1
            for action, nxt in self.successors(state):
                if nxt not in parent:
                    parent[nxt] = (state, action)
                    frontier.append(nxt)
        return None                             # no path exists

    def act(self, percept):
        if percept == self.goal:
            return None                         # goal reached: stop
        if not self.plan:
            plan = self.search(percept)
            if plan is None:
                raise RuntimeError("No path to the goal exists")
            self.plan = plan
        return self.plan.pop(0)


def run(map_text, verbose=True):
    env = Warehouse(map_text)
    agent = GoalBasedAgent(env, env.goal)
    trace = [env.percept()]
    try:
        while (action := agent.act(env.percept())) is not None:
            env.execute(action)
            trace.append(env.percept())
    except RuntimeError as e:
        if verbose:
            print("Result:", e)
        return None
    if verbose:
        grid = [row[:] for row in env.grid]
        for r, c in trace:
            if grid[r][c] == ".":
                grid[r][c] = "*"
        print("\n".join("".join(row) for row in grid))
        print(f"\nPath ({len(trace) - 1} moves, {agent.expanded} states expanded during planning):")
        print(" -> ".join(map(str, trace)))
    return trace


trace = run(WAREHOUSE)

#####################
#S***.#************G#
#.##****##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################

Path (20 moves, 58 states expanded during planning):
(1, 1) -> (1, 2) -> (1, 3) -> (1, 4) -> (2, 4) -> (2, 5) -> (2, 6) -> (2, 7) -> (1, 7) -> (1, 8) -> (1, 9) -> (1, 10) -> (1, 11) -> (1, 12) -> (1, 13) -> (1, 14) -> (1, 15) -> (1, 16) -> (1, 17) -> (1, 18) -> (1, 19)


### Testing the generated program

The agent should (1) only step on free cells, (2) move one square at a time, (3) end at `G`, (4) find a *shortest* path, and (5) report failure cleanly if `G` is unreachable.

In [2]:
env = Warehouse(WAREHOUSE)

# 1-3: validity of the executed trajectory
assert trace[0] == env.start and trace[-1] == env.goal
assert all(env.is_free(p) for p in trace)
assert all(abs(a[0] - b[0]) + abs(a[1] - b[1]) == 1 for a, b in zip(trace, trace[1:]))

# 4: optimality - brute-force check with an independent distance computation (flood fill)
def flood_distances(env):
    dist = {env.start: 0}
    q = deque([env.start])
    while q:
        r, c = q.popleft()
        for dr, dc in MOVES.values():
            n = (r + dr, c + dc)
            if env.is_free(n) and n not in dist:
                dist[n] = dist[(r, c)] + 1
                q.append(n)
    return dist

assert len(trace) - 1 == flood_distances(env)[env.goal]
print("Path valid and shortest:", len(trace) - 1, "moves")

# 5: unreachable goal
BLOCKED = """
#######
#S..#G#
#...#.#
#######
"""
print("\nBlocked map:")
assert run(BLOCKED) is None

# trivial: goal next to start
print("\nTrivial map:")
assert len(run("""
####
#SG#
####
""")) == 2
print("\nAll tests passed")

Path valid and shortest: 20 moves

Blocked map:
Result: No path to the goal exists

Trivial map:
####
#SG#
####

Path (1 moves, 1 states expanded during planning):
(1, 1) -> (1, 2)

All tests passed


### Questions

1. **Did the LLM generate a working program on the first attempt?** Yes – it ran first time and found a 20-move path. "Runs" is not the same as "correct", though, so the tests above check validity, optimality (against an independent flood fill) and failure handling separately; all passed.
2. **How can the prompt be improved?** The lab's suggested prompt only asks for "a path", so a plain path-finder with no agent structure would satisfy it. Adding the explicit architecture requirement (the last line of my prompt: an agent class with a percept → plan → act loop) makes the output match the goal-based agent design from Task 2. Further improvements: state the expected output format, ask for a *shortest* path explicitly, and list the tests the program must pass (unreachable goal, trivial map).
3. **Which search algorithm did the LLM choose?** Breadth-first search.
4. **Why?** All moves cost 1, so BFS is complete and returns the path with the fewest moves; it is simple to implement correctly, and the map is tiny so memory is not a concern. A* would also work and expand fewer states on bigger maps, but adds a heuristic that is unnecessary at this size.

**Strengths of LLM-assisted development seen here:** fast, readable, well-commented code; a sensible algorithm choice with a correct justification. **Limitations:** it only does what the prompt specifies (without the architecture line, nothing forces an actual agent loop), and it does not test its own output – the validity, optimality and failure tests above were written separately.